# 01 — Search for a Google Scholar Author via SerpAPI

**SerpAPI** is a web scraping service that provides structured access to search engine results pages. Its `google_scholar_profiles` engine lets you search for authors on Google Scholar without being blocked.

This notebook shows how to:
1. Load the API key from a `.env` file
2. Search for an author by name
3. Inspect the profile results
4. Select the best match and extract the Google Scholar author ID

**Prerequisite:** Create a `.env` file in the repository root (copy `.env.example`) and add your SerpAPI key.  
**SerpAPI docs:** https://serpapi.com/google-scholar-profiles-api

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from serpapi import GoogleSearch

In [ ]:
load_dotenv('../.env')  # load SERPAPI_KEY from the .env file in the repo root
SERPAPI_KEY = os.getenv('SERPAPI_KEY')

if not SERPAPI_KEY or SERPAPI_KEY == 'your_serpapi_key_here':
    raise ValueError('SERPAPI_KEY not set. Copy .env.example to .env and add your key.')
print('API key loaded.')

## 1. Search for a single author by name

In [ ]:
author_name = 'Rasheed Omobolaji Alabi'

params = {
    'engine': 'google_scholar_profiles',
    'mauthors': author_name,
    'api_key': SERPAPI_KEY,
}

search = GoogleSearch(params)
results = search.get_dict()

profiles = results.get('profiles', [])
print(f'Found {len(profiles)} profile(s) for "{author_name}"')

In [ ]:
# Inspect the raw result for the first profile
if profiles:
    print('First profile result:')
    print(json.dumps(profiles[0], indent=2))

## 2. Display all candidate profiles

In [ ]:
for i, profile in enumerate(profiles, 1):
    affiliations = profile.get('affiliations', 'N/A')
    interests = ', '.join(profile.get('interests', [{'title': 'N/A'}])[:3])  # first 3 interests (if available)
    # interests may be a list of dicts or strings depending on API version
    if isinstance(profile.get('interests', [None])[0], dict):
        interests = ', '.join(p['title'] for p in profile.get('interests', [])[:3])
    print(f"{i}. {profile.get('name')}")
    print(f"   Author ID   : {profile.get('author_id')}")
    print(f"   Affiliation : {affiliations}")
    print(f"   Interests   : {interests}")
    print(f"   Link        : {profile.get('link')}")
    print()

## 3. Select the best match

In [ ]:
# The first result is usually the best match when searching by full name.
# In a batch pipeline you can add logic to pick by affiliation keyword.

if profiles:
    best = profiles[0]
    author_id = best['author_id']
    print(f'Selected author : {best["name"]}')
    print(f'Google Scholar ID: {author_id}')
    print()
    print('Use this author_id in notebook 02 to fetch full metrics.')
else:
    print('No profiles found — try a different name spelling.')
    author_id = None

## 4. Helper function for reuse in later notebooks

In [ ]:
def get_scholar_author_id(name: str, api_key: str) -> str | None:
    """Search Google Scholar for an author by name and return their profile ID.
    Returns the first match, or None if no profiles found.
    """
    params = {
        'engine': 'google_scholar_profiles',
        'mauthors': name,
        'api_key': api_key,
    }
    search = GoogleSearch(params)
    profiles = search.get_dict().get('profiles', [])
    return profiles[0]['author_id'] if profiles else None


# Example: look up a second author from the dataset
test_name = 'Lamiaa Abdel-Hamid'
test_id = get_scholar_author_id(test_name, SERPAPI_KEY)
print(f'Author ID for "{test_name}": {test_id}')